# Training, Optimization & Learning Curves

In this notebook you build a training procedure yourself. It consists of three functions, with a clear
division of labour:

- `train()`: one pass over a data loader with a weight update per batch. Returns the average loss. Given.
- `evaluate()`: the same pass without weight updates. Returns the average loss and the accuracy. Given,
  and you extend it once, with precision and recall.
- `fit()`: the whole trajectory. It builds the model, the optimizer and the data loader, calls the two
  functions above once per epoch, and collects the results. **This is the function you grow.**

It starts as the barest version that works, and in every section you add one thing and run one experiment
that shows what that thing does. In most sections that one thing goes into `fit()`; in the learning-rate
section the new `fit()` code is given, you extend `evaluate()`, and `fit()` only has to record the new numbers.

- Training knobs: epochs, batches and steps, learning rate, SGD vs. Adam
- Learning curves: optimization, capacity, or overfitting?
- Early stopping: the first, nearly free remedy

**How each section works.** A short explanation, then a cell marked `YOUR TURN` that contains the code
as the previous section left it. Change it as the slides ask, run it, and then run the experiment
below it. The full solution is folded away under every exercise: open it only after you have tried, or
copy it if you are stuck, so that you can carry on with the next section.


## Setup

The same 1,000 MNIST digits and the same small CNN throughout: two convolution blocks and one dense layer,
the kind of network you built in notebook 03. Only the training settings change, so any difference you see
comes from the setting and nothing else. The training set is deliberately small: with more images this
network hardly overfits, and the second half of the notebook is about overfitting.

In [ ]:
import copy                               # deepcopy: a real copy of the weights
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import precision_score, recall_score

train_full = datasets.MNIST(root="../data/raw", train=True, download=True,
                            transform=transforms.ToTensor())
test_full = datasets.MNIST(root="../data/raw", train=False, download=True,
                           transform=transforms.ToTensor())

train_data = Subset(train_full, range(1000))      # deliberately small: overfitting has to be visible
validation_data = Subset(test_full, range(2000))


def build_model(hidden=64):
    """A small CNN: two convolution blocks, then a dense layer of `hidden` units, then the 10 scores."""
    torch.manual_seed(1)                    # same starting weights for every run
    return nn.Sequential(
        nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 1x28x28 -> 8x14x14
        nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 8x14x14 -> 16x7x7
        nn.Flatten(),                                                              # -> 784 numbers
        nn.Linear(16 * 7 * 7, hidden), nn.ReLU(),                                  # the dense layer; hidden is its width
        nn.Linear(hidden, 10),                                                     # one score per digit
    )


print(f"training images: {len(train_data)}, validation images: {len(validation_data)}")

## The starting point

First the two functions that `fit()` calls. `train()` is the inner loop you know from notebook 03, with a
shorter argument list: for every batch, a forward pass, the loss, a backward pass and one optimizer step.
`evaluate()` is the same loop without the three optimizer lines: it takes the same model, loader and loss
function, and returns the average loss and the accuracy. Keeping these two separate from the rest means
`fit()` can grow below without the loop itself getting longer.


In [ ]:
def train(model, loader, loss_function, optimizer):
    """One pass over the loader with weight updates. Returns the average loss per image."""
    model.train()
    total_loss = 0.0
    for images, labels in loader:
        optimizer.zero_grad()
        loss = loss_function(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)      # loss is the mean over the batch: times the batch size gives the sum
    return total_loss / len(loader.dataset)          # the sum over all images, divided once: an honest average


def evaluate(model, loader, loss_function):
    """One pass over the loader without weight updates. Returns the average loss per image and the accuracy."""
    model.eval()
    total_loss = correct = 0
    with torch.no_grad():                            # no gradients: faster, and nothing can change by accident
        for images, labels in loader:
            logits = model(images)
            total_loss += loss_function(logits, labels).item() * len(labels)
            correct += (logits.argmax(dim=1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


And this is `fit()`: one optimizer (Adam), one batch size, and per epoch a printed training loss,
validation loss and validation accuracy. Read it line by line before you run it, because everything
below is a change to this function.


In [ ]:
def fit(epochs=5, learning_rate=1e-3):
    model = build_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
    validation_loader = DataLoader(validation_data, batch_size=512)    # no shuffle and a large batch: no weights
                                                                       # change here, so the batch size only sets speed

    for epoch in range(1, epochs + 1):
        train_loss = train(model, train_loader, loss_function, optimizer)
        val_loss, val_accuracy = evaluate(model, validation_loader, loss_function)
        print(f"epoch {epoch}: training loss {train_loss:.3f}, validation loss {val_loss:.3f}, "
              f"validation accuracy {val_accuracy:.3f}")


fit()


Both losses fall, so the loop works. It also tells you very little: you cannot change the batch size,
and you cannot plot anything, so you cannot put two runs next to each other. The rest of the notebook
fixes that, one step at a time.


## Training knobs: epochs, batches and steps

Three words that are easy to confuse:

- an **epoch** is one full pass over the training data;
- a **batch** is the group of examples used for one weight update;
- a **step** is one weight update.

They are connected by one division:

$$\text{steps per epoch} = \left\lceil \frac{\text{training examples}}{\text{batch size}} \right\rceil$$

**Your turn.** The batch size is fixed at 64 inside the function. Make it a parameter.

In [ ]:
# YOUR TURN - exercise 1. This is fit() as the previous section left it.

def fit(epochs=5, learning_rate=1e-3):
    model = build_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
    validation_loader = DataLoader(validation_data, batch_size=512)   

    for epoch in range(1, epochs + 1):
        train_loss = train(model, train_loader, loss_function, optimizer)
        val_loss, val_accuracy = evaluate(model, validation_loader, loss_function)
        print(f"epoch {epoch}: training loss {train_loss:.3f}, validation loss {val_loss:.3f}, "
              f"validation accuracy {val_accuracy:.3f}")


In [ ]:
# Experiment: the same 5 epochs at three batch sizes.
for batch_size in [16, 64, 256]:
    steps = len(DataLoader(train_data, batch_size=batch_size))
    print(f"\nbatch size {batch_size}: {steps} steps per epoch, {steps * 5} weight updates in 5 epochs")
    fit(epochs=5, batch_size=batch_size)

<details>
<summary>Solution</summary>

```python
def fit(epochs=5, learning_rate=1e-3, batch_size=64):
    model = build_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    validation_loader = DataLoader(validation_data, batch_size=512)    # no shuffle and a large batch: no weights
                                                                       # change here, so the batch size only sets speed

    for epoch in range(1, epochs + 1):
        train_loss = train(model, train_loader, loss_function, optimizer)
        val_loss, val_accuracy = evaluate(model, validation_loader, loss_function)
        print(f"epoch {epoch}: training loss {train_loss:.3f}, validation loss {val_loss:.3f}, "
              f"validation accuracy {val_accuracy:.3f}")
```

Why the validation loader keeps 512: `evaluate()` changes no weights, so the batch size there has no effect
on the result. It only sets how many images go through the network at once, which is speed and memory.
The batch size of the training loader is a different thing: it decides how many examples each weight
update is based on, and that is what this section is about.

</details>


"5 epochs" is not a fixed amount of training: 315 updates at batch size 16, and 20 at batch size 256.
Compare the final losses with that in mind. **Comparing two settings at the same number of epochs is
only fair if the batch size is the same**; otherwise you are comparing different amounts of training.

Small batches also give noisier updates, because each one is computed from fewer examples. Large batches
give smoother steps, and fewer of them.

### How do you choose?

There is no formula, but there are habits that almost everyone follows:

- **Start at 32 or 64.** For small images and small networks, as in this course, anything between 32 and
  128 behaves well. Powers of two are a convention, not a requirement.
- **Memory sets the ceiling.** Every image in a batch sits in memory at once, with all its intermediate
  activations. An out-of-memory error means the batch is too large for your GPU or laptop: halve it.
- **Go larger for speed, smaller for noise.** A larger batch means fewer, faster steps per epoch, which is
  what you want with a GPU and a lot of data. A smaller batch gives noisier steps, and that noise often helps
  the network generalize a little better. Tiny batches (under 16) are mostly noise.
- **Batch size and learning rate belong together.** If you make the batch much larger, each step is based
  on more examples and can be bigger too. The usual starting point: double the batch size, double the
  learning rate, and then check. The next section is about that learning rate.
- **Do not choose epochs up front.** Watch the validation loss and stop when it stops improving. That is
  what *Learning curves* and *Early stopping* below are about. Five epochs here is a number for a quick
  experiment, not a recommendation.
- **Change one thing at a time.** When you compare two batch sizes, keep the learning rate and the number
  of epochs the same, and say in your report how many updates each run made.

For the image assignment: 32 or 64, chosen once, written down, and then left alone while you tune the
things that matter more.


## Training knobs: the learning rate

The learning rate is how big each step is. Too small crawls, too large jumps over anything useful. To
see that you need the numbers of several runs in one figure, and printed numbers cannot be plotted. So
from here on `fit()` collects them: a list `history` with one dictionary per epoch, returned as a
`DataFrame` with one row per epoch and one column per number. That code is given in the cell below.
Read it before you go on, because the plotting code of every experiment leans on it.

**Your turn.** Accuracy has been the only score so far, and you saw in the previous session that accuracy
can hide a lot: precision and recall say *which* mistakes a model makes. Extend `evaluate()` so that it
also returns precision and recall, computed with scikit-learn over all validation images, and record both
in the history. One thing is new: MNIST has ten classes, not two. Use `average="macro"`, which computes
the score for each digit separately and averages the ten numbers.


In [ ]:
# YOUR TURN - exercise 2. Two functions this time: evaluate() gets precision and recall, and fit() has to record them.

def evaluate(model, loader, loss_function):
    """One pass over the loader without weight updates. Returns the average loss per image and the accuracy."""
    model.eval()
    total_loss = correct = 0
    all_labels, all_predictions = [], []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images)
            total_loss += loss_function(logits, labels).item() * len(labels)
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def fit(epochs=5, learning_rate=1e-3, batch_size=64):
    model = build_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    validation_loader = DataLoader(validation_data, batch_size=512)    # no shuffle and a large batch: no weights
                                                                       # change here, so the batch size only sets speed

    history = []
    for epoch in range(1, epochs + 1):
        train_loss = train(model, train_loader, loss_function, optimizer)
        val_loss, val_accuracy = evaluate(model, validation_loader, loss_function)
        history.append({"epoch": epoch, "train_loss": train_loss,
                        "val_loss": val_loss, "val_accuracy": val_accuracy})
    return pd.DataFrame(history)


In [ ]:
# Experiment: four learning rates, three factors of ten or more apart.
fig, ax = plt.subplots(figsize=(8.5, 4.2))
for learning_rate in [1e-5, 1e-3, 1e-2, 1.0]:
    history = fit(epochs=15, learning_rate=learning_rate)
    ax.plot(history["epoch"], history["train_loss"], marker="o", label=f"lr = {learning_rate:g}")
    final = history.iloc[-1]
    print(f"lr = {learning_rate:g}: training loss after 15 epochs {final['train_loss']:.4f}, "
          f"validation accuracy {final['val_accuracy']:.3f}, precision {final['val_precision']:.3f}, recall {final['val_recall']:.3f}")
ax.set(xlabel="epoch", ylabel="training loss", title="Four learning rates, Adam", yscale="log")
ax.legend()
plt.tight_layout()
plt.show()

<details>
<summary>Solution</summary>

```python
def evaluate(model, loader, loss_function):
    """One pass over the loader without weight updates. Returns the average loss per image, the accuracy,
    and the macro-averaged precision and recall."""
    model.eval()
    total_loss = correct = 0
    all_labels, all_predictions = [], []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images)
            total_loss += loss_function(logits, labels).item() * len(labels)
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            all_labels.append(labels)
            all_predictions.append(predictions)

    y_true = torch.cat(all_labels).numpy()               # one array with every label, in loader order
    y_pred = torch.cat(all_predictions).numpy()          # and one with every prediction
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    return total_loss / len(loader.dataset), correct / len(loader.dataset), precision, recall


def fit(epochs=5, learning_rate=1e-3, batch_size=64):
    model = build_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    validation_loader = DataLoader(validation_data, batch_size=512)    # no shuffle and a large batch: no weights
                                                                       # change here, so the batch size only sets speed

    history = []
    for epoch in range(1, epochs + 1):
        train_loss = train(model, train_loader, loss_function, optimizer)
        val_loss, val_accuracy, val_precision, val_recall = evaluate(model, validation_loader, loss_function)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "val_accuracy": val_accuracy, "val_precision": val_precision, "val_recall": val_recall})
    return pd.DataFrame(history)
```

`average="macro"` computes precision (or recall) for each of the ten digits separately and takes the plain
mean, so every digit counts equally. `zero_division=0` keeps scikit-learn quiet when a model never
predicts some digit at all, which happens at a learning rate of 1e-5.

</details>


Look for these shapes in your figure, and write down which learning rate produced which:

- **almost flat**: nothing is broken, the steps are simply too small to get anywhere in fifteen epochs;
- **steady fall**: what a healthy curve looks like;
- **fast but jumpy**: usable, close to the edge;
- **far above the rest, or not falling at all**: the step is larger than the distance to anything
  useful, so each update lands somewhere worse.

The printed precision and recall tell the same story from the other side. At a learning rate of 1 the
network predicts the same digit for every image: accuracy about 10%, recall 0.10 (one digit found, the
other nine never), precision 0.01 (averaged over ten digits, nine of which have no true positives).

In your own work: a flat curve means too small, a curve that rises, jumps around or turns into `nan`
means too large. Change the learning rate by factors of ten, not by 10%.

## Training knobs: SGD vs. Adam

So far the optimizer has been Adam, without saying what that is. The plain alternative is **SGD**: one
learning rate for every parameter, forever. Adam does two extra things:

1. **Momentum**: it keeps a running average of recent gradients, so the path is smoothed.
2. **Per-parameter scaling**: it divides each update by a running average of that parameter's recent
   gradient size, so parameters with tiny gradients still move and those with huge ones do not explode.

**Your turn.** Let the caller choose the optimizer.

In [ ]:
# YOUR TURN - exercise 3. This is fit() as the previous section left it.

def fit(epochs=5, learning_rate=1e-3, batch_size=64):
    model = build_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    validation_loader = DataLoader(validation_data, batch_size=512)    # no shuffle and a large batch: no weights
                                                                       # change here, so the batch size only sets speed

    history = []
    for epoch in range(1, epochs + 1):
        train_loss = train(model, train_loader, loss_function, optimizer)
        val_loss, val_accuracy, val_precision, val_recall = evaluate(model, validation_loader, loss_function)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "val_accuracy": val_accuracy, "val_precision": val_precision, "val_recall": val_recall})
    return pd.DataFrame(history)


In [ ]:
# Experiment: each optimizer at a learning rate that suits it.
sgd_history = fit(epochs=15, learning_rate=0.05, optimizer_name="sgd")
adam_history = fit(epochs=15, learning_rate=0.001, optimizer_name="adam")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sgd_history["epoch"], sgd_history["train_loss"], marker="o", label="SGD, lr = 0.05")
ax.plot(adam_history["epoch"], adam_history["train_loss"], marker="o", label="Adam, lr = 0.001")
ax.set(xlabel="epoch", ylabel="training loss", title="SGD and Adam on the same network")
ax.legend()
plt.tight_layout()
plt.show()

<details>
<summary>Solution</summary>

```python
def fit(epochs=5, learning_rate=1e-3, batch_size=64, optimizer_name="adam"):
    model = build_model()
    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    validation_loader = DataLoader(validation_data, batch_size=512)    # no shuffle and a large batch: no weights
                                                                       # change here, so the batch size only sets speed

    history = []
    for epoch in range(1, epochs + 1):
        train_loss = train(model, train_loader, loss_function, optimizer)
        val_loss, val_accuracy, val_precision, val_recall = evaluate(model, validation_loader, loss_function)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "val_accuracy": val_accuracy, "val_precision": val_precision, "val_recall": val_recall})
    return pd.DataFrame(history)
```

</details>


Note the learning rates in the legend: Adam at 0.001 against SGD at 0.05. They are not comparable
numbers. Adam rescales its updates internally, so `1e-3` is its usual starting point, while SGD
typically needs something one or two orders of magnitude larger. Try `optimizer_name="sgd"` with
`learning_rate=0.001` to see what happens when you reuse Adam's value.

Adam is an excellent default and not a magic fix: it still needs its learning rate tuned.

## Learning curves

Everything so far plotted the training loss alone, and a model that memorizes its training set looks
excellent by that number. The validation loss has been there since the first cell, as a number per epoch.
A **learning curve** is the two numbers plotted together, and the gap between the two curves is the
diagnosis.

**Your turn.** Make the width of the dense layer a parameter, so that you can compare a small and a
large model. The experiment below then draws the learning curves of three runs.


In [ ]:
# YOUR TURN - exercise 4. This is fit() as the previous section left it.

def fit(epochs=5, learning_rate=1e-3, batch_size=64, optimizer_name="adam"):
    model = build_model()
    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    validation_loader = DataLoader(validation_data, batch_size=512)    # no shuffle and a large batch: no weights
                                                                       # change here, so the batch size only sets speed

    history = []
    for epoch in range(1, epochs + 1):
        train_loss = train(model, train_loader, loss_function, optimizer)
        val_loss, val_accuracy, val_precision, val_recall = evaluate(model, validation_loader, loss_function)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "val_accuracy": val_accuracy, "val_precision": val_precision, "val_recall": val_recall})
    return pd.DataFrame(history)


In [ ]:
# Experiment: three runs, three different problems.
runs = {
    "learning rate 1e-5": fit(epochs=25, learning_rate=1e-5),
    "64 hidden units": fit(epochs=25),
    "512 hidden units": fit(epochs=25, hidden=512),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (title, history) in zip(axes, runs.items()):
    ax.plot(history["epoch"], history["train_loss"], marker="o", label="train")
    ax.plot(history["epoch"], history["val_loss"], marker="o", label="validation")
    ax.set(xlabel="epoch", title=title)
    ax.legend()
axes[0].set_ylabel("loss")
plt.tight_layout()
plt.show()

for title, history in runs.items():
    final = history.iloc[-1]
    best_epoch = history.loc[history["val_loss"].idxmin(), "epoch"]
    print(f"{title:<19} train loss {final['train_loss']:.3f}  val loss {final['val_loss']:.3f}  "
          f"val accuracy {final['val_accuracy']:.3f}  precision {final['val_precision']:.3f}  "
          f"recall {final['val_recall']:.3f}  best epoch {best_epoch}")

<details>
<summary>Solution</summary>

```python
def fit(epochs=5, learning_rate=1e-3, batch_size=64, optimizer_name="adam", hidden=64):
    model = build_model(hidden)
    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    validation_loader = DataLoader(validation_data, batch_size=512)    # no shuffle and a large batch: no weights
                                                                       # change here, so the batch size only sets speed

    history = []
    for epoch in range(1, epochs + 1):
        train_loss = train(model, train_loader, loss_function, optimizer)
        val_loss, val_accuracy, val_precision, val_recall = evaluate(model, validation_loader, loss_function)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "val_accuracy": val_accuracy, "val_precision": val_precision, "val_recall": val_recall})
    return pd.DataFrame(history)
```

</details>


**The diagnostic question:** is the gap between the two curves large, and is validation getting *worse*?

- **Small gap, both curves high and still falling: underfitting.** The model is not learning fast
  enough. Train longer, raise the learning rate, or add capacity.
- **Large gap, training loss still falling while validation loss stops falling or turns upward:
  overfitting.** Everything after that point is computation that makes the model no better, and usually
  worse, on new data.

Now read the printed lines. Which model has the lowest training loss, and which has the best validation
accuracy? If the larger model overfits harder *and* scores better on validation, shrinking it would
throw away the better model. Overfitting describes the shape of the curves; it tells you **when to
stop**, not which model to keep. The "best epoch" column says when.

## Early stopping

The 512-unit run had its best validation loss around epoch 15 and did not improve after that, while
its training loss kept falling. Early stopping automates the obvious response: watch the validation
loss, and stop when it has not improved for a while.

Two details make it work:

- **Patience**: do not stop at the first bad epoch, because the curve is noisy; wait a few.
- **Restore the best weights**: otherwise you keep the model from the last epoch, which is worse than
  the best one.

**Your turn.** This is the largest change in the notebook, so take it one step at a time. All of it
goes into `fit()`: the epoch loop lives there, and that is the loop that has to stop. A copy of
the weights is `copy.deepcopy(model.state_dict())`, and you put one back with `model.load_state_dict(...)`.
The `deepcopy` matters: `model.state_dict()` on its own only points at the live weights, so without it your
"saved" weights would keep changing with every training step.

In [ ]:
# YOUR TURN - exercise 5. This is fit() as the previous section left it.


def fit(epochs=5, learning_rate=1e-3, batch_size=64, optimizer_name="adam", hidden=64):
    model = build_model(hidden)
    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    validation_loader = DataLoader(validation_data, batch_size=512)    # no shuffle and a large batch: no weights
                                                                       # change here, so the batch size only sets speed

    history = []
    for epoch in range(1, epochs + 1):
        train_loss = train(model, train_loader, loss_function, optimizer)
        val_loss, val_accuracy, val_precision, val_recall = evaluate(model, validation_loader, loss_function)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "val_accuracy": val_accuracy, "val_precision": val_precision, "val_recall": val_recall})
    return pd.DataFrame(history)


In [ ]:
# Experiment: the overfitting model, allowed 40 epochs, with a patience of 5.
history = fit(epochs=40, hidden=512, patience=5)

best_epoch = history.loc[history["val_loss"].idxmin(), "epoch"]
last_epoch = history["epoch"].iloc[-1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history["epoch"], history["val_loss"], marker="o", label="validation loss")
ax.axvline(best_epoch, color="green", linestyle="--", label=f"best epoch ({best_epoch})")
ax.axvline(last_epoch, color="red", linestyle=":", label=f"stopped here ({last_epoch})")
ax.set(xlabel="epoch", ylabel="validation loss", title="Early stopping keeps the best model, not the last")
ax.legend()
plt.tight_layout()
plt.show()

<details>
<summary>Solution</summary>

```python
def fit(epochs=5, learning_rate=1e-3, batch_size=64, optimizer_name="adam", hidden=64,
        patience=None):
    model = build_model(hidden)
    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    validation_loader = DataLoader(validation_data, batch_size=512)    # no shuffle and a large batch: no weights
                                                                       # change here, so the batch size only sets speed

    best_loss, best_epoch, best_weights, waited = float("inf"), 0, None, 0
    history = []
    for epoch in range(1, epochs + 1):
        train_loss = train(model, train_loader, loss_function, optimizer)
        val_loss, val_accuracy, val_precision, val_recall = evaluate(model, validation_loader, loss_function)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "val_accuracy": val_accuracy, "val_precision": val_precision, "val_recall": val_recall})

        if patience is not None:
            if val_loss < best_loss:
                best_loss, best_epoch, waited = val_loss, epoch, 0
                # deepcopy makes a real copy. Without it, best_weights would point at the live weights
                # and keep changing with every step: at the end you would restore the last epoch, not the best.
                best_weights = copy.deepcopy(model.state_dict())
            else:
                waited += 1
                if waited >= patience:
                    print(f"stopped at epoch {epoch}: no improvement for {patience} epochs")
                    break

    if patience is not None:
        model.load_state_dict(best_weights)          # restore the best, not the last
        restored_loss, *_ = evaluate(model, validation_loader, loss_function)
        print(f"restored the weights of epoch {best_epoch}: validation loss {restored_loss:.3f}")
    return pd.DataFrame(history)
```

The solution also evaluates the model once more after restoring, and prints that loss. It should equal
the lowest point of the curve: that is the proof that you kept the best model and not the last one.

</details>


The distance between the green and the red line is the patience: training continued for a few more
epochs to confirm that the improvement had really stopped, and then the weights from the green line
were put back.

One caution: the validation loss you stopped on is now slightly optimistic, because you used it to
choose. If the number matters, keep a separate test set for the final report.

## Knowledge check

1. You train for 20 epochs with batch size 32 on 1,000 examples. How many weight updates is that?
2. Why can you not reuse an SGD learning rate for Adam?
3. Both curves are high and still falling. Underfitting or overfitting, and what do you change?
4. What two things does early stopping need in order to be useful?
5. A model predicts the same digit for every MNIST image. Roughly what are its accuracy, macro recall and
   macro precision?

<details>
<summary>Suggested answers</summary>

1. 1,000 / 32 = 32 steps per epoch (rounded up), so 640 updates.
2. Because Adam rescales each update by a running average of that parameter's gradient size, so its
   learning rate is on a different scale. About 1e-3 for Adam; one or two orders of magnitude more for
   SGD.
3. Underfitting. Train longer, raise the learning rate, or add capacity.
4. Patience, so that a single noisy epoch does not stop training; and restoring the best weights, so
   that you keep the model you found rather than the last one you trained.
5. Accuracy about 10%, one digit in ten. Macro recall 0.10: recall is 1 for that digit and 0 for the other
   nine, mean 0.1. Macro precision about 0.01: precision is 0.1 for that digit and 0 for the nine digits it
   never predicts, mean 0.01.

</details>

## Recap

- An epoch is not a fixed amount of training: steps per epoch = examples / batch size.
- Search the learning rate by factors of ten, and recognise too small and too large from the curve.
- Adam adds momentum and per-parameter scaling, and needs a different learning rate from SGD.
- A learning curve answers one question first: is the gap large, and is validation getting worse?
- Early stopping needs patience and the best weights restored. Overfitting says when to stop, not which
  model to keep.
- Record more than accuracy. Precision and recall come from the same predictions; with more than two
  classes, `average="macro"` gives every class the same weight.
- Keep the pass over the data, the scoring and the whole run in three functions. `fit()` grew from 12 lines
  to 34 and the inner loop never changed.

Your `fit()` now has six parameters, and you added four of them yourself.